# 06 - Simulacion contrafactual

Hasta aqui el trabajo identifica la carencia y propone un refuerzo. Queda la pregunta
obvia: **habria evitado el descenso?**

La respuesta honesta es que no se puede estimar, y el camino hasta ahi es parte del
resultado. Este notebook lo recorre en cuatro pasos:

1. Intentar medir empiricamente la relacion entre la anticipacion de los centrales y el
   xG concedido en el centro del area. **Queda descartado.**
2. Simular la temporada remate a remate y barrer distintos recortes en esa zona.
3. Acotar el techo del cambio con los minutos reales del central al que sustituiria.
4. Dibujar el resultado.

Lee `disparos_laliga_2526_limpio.csv` y `metricas_jugadores_laliga_2526.csv`, y guarda
`images/simulacion_contrafactual.png` y su version oscura.

## 1. El intento de estimar el efecto

Si la anticipacion de los centrales redujera el xG concedido en el centro del area,
tendria que verse comparando los veinte equipos: los que tienen centrales que interceptan
mas deberian conceder menos en esa zona.

Se construye una tabla por equipo con el xG concedido alli en juego en curso y la tasa de
intercepciones de sus centrales ponderada por minutos, y se ajusta una regresion lineal.

In [1]:
# Instalacion de dependencias
# !pip install pandas numpy scipy

# Librerias de manejo de datos
import os
import numpy as np
import pandas as pd
# Distribucion t de Student para calcular los p-valores de la regresion
from scipy import stats

# Deteccion automatica de la raiz del proyecto
BASE = ".." if os.path.isdir("../data") else "."
# Carpeta de datos
DATOS = os.path.join(BASE, "data")

# Carga de los remates de la temporada con su xG
d = pd.read_csv(os.path.join(DATOS, "disparos_laliga_2526_limpio.csv"))
# Carga de las metricas individuales de los jugadores
m = pd.read_csv(os.path.join(DATOS, "metricas_jugadores_laliga_2526.csv"))

# Lista ordenada de los equipos tal como aparecen en el fichero de remates
equipos = sorted(d["team"].dropna().unique())
# Diccionario para guardar la pareja local-visitante de cada partido
pares = {}
# Recorrido de todos los partidos de la temporada
for g in d["game"].dropna().unique():
    # El nombre del partido viene como "fecha Local-Visitante"
    nom = g.split(" ", 1)[1]
    # Equipos cuyo nombre encaja al principio de la cadena
    loc = [e for e in equipos if nom.startswith(e + "-")]
    # Equipos cuyo nombre encaja al final de la cadena
    vis = [e for e in equipos if nom.endswith("-" + e)]
    # Solo se guarda el partido si se identifican los dos equipos
    if loc and vis:
        pares[g] = (max(loc, key=len), max(vis, key=len))

# Equipo local de cada remate
d["local"] = d["game"].map(lambda g: pares.get(g, (None, None))[0])
# Equipo visitante de cada remate
d["visitante"] = d["game"].map(lambda g: pares.get(g, (None, None))[1])
# Equipo que concede el remate: el rival del que dispara
d["defensor"] = np.where(d["team"] == d["local"], d["visitante"], d["local"])

# Las coordenadas pueden venir en escala 0-1 o 0-100 segun la fuente
escala = 100.0 if d["location_x"].max() <= 1.5 else 1.0
# Coordenada longitudinal normalizada a 0-100
d["x100"] = d["location_x"] * escala
# Coordenada transversal normalizada a 0-100
d["y100"] = d["location_y"] * escala

# Limites de la zona analizada: centro del area grande
AREA_X, CEN_Y0, CEN_Y1 = 84.3, 33.3, 66.7
# Marca de los remates que caen en el centro del area
d["centro_area"] = (d["x100"] >= AREA_X) & (d["y100"].between(CEN_Y0, CEN_Y1))
# Marca de los remates producidos en juego en curso
d["juego_abierto"] = d["situation"] == "Open Play"

# xG concedido en el centro del area en juego en curso, por equipo
xga_centro = (d[d["centro_area"] & d["juego_abierto"]]
              .groupby("defensor")["xg"].sum())
# xG concedido en total, por equipo
xga_total = d.groupby("defensor")["xg"].sum()
# xG generado en total, por equipo
xg_favor = d.groupby("team")["xg"].sum()

# Centrales de la competicion con minutos registrados
centrales = m[(m["posicion"] == "central") & (m["minutes"] > 0)]
# Tasa de intercepciones de los centrales de cada equipo, ponderada por minutos
int_centrales = (centrales.groupby("team")
                 .apply(lambda t: t["Interception"].sum() / t["minutes"].sum() * 90))

# Tabla final con una fila por equipo
tabla = pd.DataFrame({
    "xga_centro": xga_centro,
    "xga_total": xga_total,
    "xg_favor": xg_favor,
    "int_centrales": int_centrales,
}).dropna()
# Paso del acumulado de temporada a media por partido
tabla["xga_centro_pp"] = tabla["xga_centro"] / 38

# Intento de incorporar la altura defensiva como variable de control
ruta_altura = os.path.join(DATOS, "altura_defensiva_laliga_2526.csv")
# Variable que indicara si la altura se ha podido cruzar
tiene_altura = False
# Solo se intenta si el fichero existe
if os.path.exists(ruta_altura):
    # Carga del fichero de altura defensiva
    a = pd.read_csv(ruta_altura)
    # Posibles nombres de la columna de equipo
    col_eq = next((c for c in ["team", "equipo", "squad", "Equipo"] if c in a.columns), None)
    # Posibles nombres de la columna de altura
    col_al = next((c for c in a.columns
                   if c != col_eq and "altura" in c.lower()), None)
    # Si se identifican las dos columnas se cruza con la tabla
    if col_eq and col_al:
        tabla = tabla.join(a.set_index(col_eq)[col_al].rename("altura"), how="left")
        # Se marca como disponible solo si no quedan huecos
        tiene_altura = tabla["altura"].notna().all()
    # Si no se identifican, se avisa para mirar el fichero a mano
    else:
        print("No se han identificado las columnas de altura. Columnas:", list(a.columns))

# Impresion de la tabla ordenada de mas a menos xG concedido en la zona
print("=== TABLA POR EQUIPO ===")
print(tabla.sort_values("xga_centro_pp", ascending=False)
      .round(3).to_string())

# Variable dependiente: xG concedido en el centro del area por partido
y = tabla["xga_centro_pp"].values
# Variables explicativas segun se haya podido cruzar la altura o no
if tiene_altura:
    X = np.column_stack([np.ones(len(tabla)),
                         tabla["int_centrales"].values,
                         tabla["altura"].values])
    nombres = ["constante", "int_centrales", "altura"]
else:
    X = np.column_stack([np.ones(len(tabla)),
                         tabla["int_centrales"].values])
    nombres = ["constante", "int_centrales"]

# Estimacion de los coeficientes por minimos cuadrados
beta = np.linalg.lstsq(X, y, rcond=None)[0]
# Valores predichos por el modelo
pred = X @ beta
# Residuos del ajuste
res = y - pred
# Grados de libertad restantes
gl = len(y) - X.shape[1]
# Varianza residual estimada
s2 = (res @ res) / gl
# Matriz de varianzas y covarianzas de los coeficientes
cov = s2 * np.linalg.inv(X.T @ X)
# Errores estandar de cada coeficiente
ee = np.sqrt(np.diag(cov))
# Estadistico t de cada coeficiente
t = beta / ee
# P-valor bilateral de cada coeficiente
p = 2 * (1 - stats.t.cdf(np.abs(t), gl))
# Coeficiente de determinacion del modelo
r2 = 1 - (res @ res) / ((y - y.mean()) @ (y - y.mean()))

# Impresion de los resultados de la regresion
print("\n=== REGRESION: xG concedido en el centro del area por partido ===")
print(f"n = {len(y)} equipos, R2 = {r2:.3f}, gl = {gl}")
# Recorrido de los coeficientes estimados
for n_, b_, e_, t_, p_ in zip(nombres, beta, ee, t, p):
    print(f"  {n_:<15} coef = {b_:>8.4f}   ee = {e_:>7.4f}   t = {t_:>6.2f}   p = {p_:.4f}")

# Correlacion simple entre intercepciones de los centrales y xG concedido
r_simple = np.corrcoef(tabla["int_centrales"], tabla["xga_centro_pp"])[0, 1]
print(f"\nCorrelacion simple intercepciones-xGA centro: {r_simple:.3f}")

# Valores del Mallorca para situarlo dentro de la distribucion
print("\n=== MALLORCA ===")
print(tabla.loc[[i for i in tabla.index if "Mallorca" in i]].round(3).to_string())

=== TABLA POR EQUIPO ===
                 xga_centro  xga_total  xg_favor  int_centrales  xga_centro_pp  altura
Levante              44.758     68.074    60.815          1.235          1.178   32.23
Elche                44.033     69.747    48.694          1.232          1.159   32.00
Real Oviedo          41.195     68.350    41.946          0.963          1.084   30.71
Girona               40.229     66.407    53.057          1.157          1.059   30.36
Espanyol             39.596     64.385    52.904          0.839          1.042   33.24
Sevilla              38.859     62.012    40.299          1.245          1.023   32.91
Mallorca             38.591     66.357    49.185          0.664          1.016   29.80
Real Sociedad        36.890     61.612    60.586          1.166          0.971   33.65
Barcelona            33.034     51.657   101.014          1.169          0.869   36.49
Athletic Club        31.239     48.959    56.667          1.303          0.822   37.73
Rayo Vallecano    

**La via queda descartada.** R2 de 0,040, coeficiente de 0,1208 con p-valor de 0,397 y
signo contrario al esperado: a mas intercepciones, algo mas de xG concedido.

La razon se ve en la propia tabla. El Real Madrid tiene 0,596 intercepciones de sus
centrales y es el que menos concede en el centro del area; el Alaves tiene 0,353 y esta
entre los que menos. Lo que domina la comparacion entre equipos no es como defienden los
centrales, sino cuanto remata el rival, y eso depende del nivel del equipo y de la
posesion. El efecto que se busca queda sepultado debajo.

Es el mismo resultado que da el modelo supervisado del notebook 04 y la correlacion del
notebook 02. **Sin coeficiente no hay prediccion posible**, asi que no se estima el efecto
del fichaje.

## 2. La simulacion de la temporada

Se invierte la pregunta: en lugar de calcular cuanto habria recortado el refuerzo, se
calcula **cuanto habria hecho falta recortar** para cambiar el resultado deportivo.

La temporada se simula remate a remate: cada disparo se resuelve como un ensayo de
Bernoulli con su propio xG, se suman los goles de cada partido y se reparten los puntos.
20.000 temporadas por escenario. Los escenarios alternativos reducen un porcentaje fijo
del xG concedido en el centro del area en juego en curso y dejan intacto todo lo demas.

El umbral de permanencia se fija en 43 puntos, uno mas que Levante y Osasuna, manteniendo
fijos los resultados del resto de la liga.

In [2]:
# Semilla fija para que la simulacion sea reproducible
rng = np.random.default_rng(42)
# Numero de temporadas simuladas en cada escenario
N_SIM = 20000
# Nombre del equipo tal como aparece en el fichero de remates
EQ = "Mallorca"

# Partidos en los que participa el Mallorca, como local o como visitante
partidos = [g for g, (l, v) in pares.items() if EQ in (l, v)]
# Comprobacion de que la temporada esta completa
print(f"Partidos del {EQ}: {len(partidos)}")

# Listas con los datos de cada partido
xg_favor_p, xg_contra_p, zona_p = [], [], []
# Recorrido de los 38 partidos
for g in partidos:
    # Remates de ese partido
    sub = d[d["game"] == g]
    # xG de cada remate del Mallorca
    xg_favor_p.append(sub.loc[sub["team"] == EQ, "xg"].values)
    # Remates que concede el Mallorca en ese partido
    con = sub[sub["defensor"] == EQ]
    # xG de cada remate concedido
    xg_contra_p.append(con["xg"].values)
    # Marca de los remates concedidos que caen en la zona analizada
    zona_p.append((con["centro_area"] & con["juego_abierto"]).values)

# Comprobacion de que los totales coinciden con la tabla por equipo
print(f"xG a favor: {sum(a.sum() for a in xg_favor_p):.3f}")
print(f"xG en contra: {sum(a.sum() for a in xg_contra_p):.3f}")
print(f"xG en contra en la zona: {sum(a[z].sum() for a, z in zip(xg_contra_p, zona_p)):.3f}")


# Funcion que simula N_SIM temporadas aplicando un recorte a la zona
def simular(recorte):
    # Acumulador de puntos de cada temporada simulada
    puntos = np.zeros(N_SIM, dtype=np.int32)
    # Recorrido de los 38 partidos
    for xf, xc, z in zip(xg_favor_p, xg_contra_p, zona_p):
        # Copia del xG concedido para no alterar los datos originales
        xc_mod = xc.copy()
        # Reduccion del xG solo en los remates de la zona analizada
        xc_mod[z] = xc_mod[z] * (1 - recorte)
        # Goles a favor: cada remate entra con probabilidad igual a su xG
        gf = (rng.random((N_SIM, len(xf))) < xf).sum(axis=1)
        # Goles en contra con el xG ya ajustado
        gc = (rng.random((N_SIM, len(xc_mod))) < xc_mod).sum(axis=1)
        # Tres puntos por victoria
        puntos += np.where(gf > gc, 3, 0)
        # Un punto por empate
        puntos += np.where(gf == gc, 1, 0)
    # Distribucion de puntos de las temporadas simuladas
    return puntos


# Recortes que se van a probar, de 0 % a 60 %
recortes = np.arange(0, 0.65, 0.05)
# Lista donde se guardan los resultados de cada escenario
filas = []
# Recorrido de los escenarios
for r in recortes:
    # Simulacion completa con ese recorte
    pts = simular(r)
    # Resultados resumidos del escenario
    filas.append({
        "recorte": f"{r*100:.0f} %",
        "media": pts.mean(),
        "mediana": np.median(pts),
        "p5": np.percentile(pts, 5),
        "p95": np.percentile(pts, 95),
        "prob_43": (pts >= 43).mean() * 100,
    })

# Tabla con todos los escenarios simulados
esc = pd.DataFrame(filas)
print("\n=== ESCENARIOS (20.000 temporadas cada uno) ===")
print("Umbral de permanencia: 43 puntos, uno mas que Levante y Osasuna,")
print("manteniendo fijos los resultados del resto de la liga.\n")
print(esc.round(2).to_string(index=False))

# Validacion: el escenario sin recorte debe parecerse a los puntos esperados reales
print(f"\nValidacion del escenario base: {esc.iloc[0]['media']:.2f} puntos simulados")
print("frente a 39,79 puntos esperados de Understat.")

# Exceso de goles esperados ya medido en el trabajo
EXCESO = 7.4
# xG total concedido por el Mallorca en la zona analizada
zona_total = sum(a[z].sum() for a, z in zip(xg_contra_p, zona_p))
# Porcentaje de la zona que representa ese exceso
print(f"\nReferencia: borrar por completo el exceso de {EXCESO} goles esperados")
print(f"equivale a un recorte del {EXCESO / zona_total * 100:.1f} % de esa zona.")

Partidos del Mallorca: 38


xG a favor: 49.185
xG en contra: 66.357
xG en contra en la zona: 38.591



=== ESCENARIOS (20.000 temporadas cada uno) ===
Umbral de permanencia: 43 puntos, uno mas que Levante y Osasuna,
manteniendo fijos los resultados del resto de la liga.

recorte  media  mediana   p5  p95  prob_43
    0 %  40.65     41.0 31.0 51.0    37.39
    5 %  41.55     41.0 32.0 52.0    43.00
   10 %  42.58     43.0 33.0 53.0    50.06
   15 %  43.55     44.0 34.0 54.0    56.48
   20 %  44.56     44.0 34.0 55.0    62.72
   25 %  45.63     46.0 35.0 56.0    68.84
   30 %  46.58     46.0 37.0 57.0    73.68
   35 %  47.73     48.0 38.0 58.0    79.46
   40 %  48.81     49.0 39.0 59.0    84.30
   45 %  49.86     50.0 40.0 60.0    88.06
   50 %  51.09     51.0 41.0 62.0    91.16
   55 %  52.22     52.0 42.0 63.0    94.01
   60 %  53.37     53.0 43.0 64.0    95.62

Validacion del escenario base: 40.65 puntos simulados
frente a 39,79 puntos esperados de Understat.

Referencia: borrar por completo el exceso de 7.4 goles esperados
equivale a un recorte del 19.2 % de esa zona.


**La simulacion se valida**: el escenario base da 40,65 puntos esperados frente a los
39,79 que publica Understat, un 2,2 % de diferencia. Es suficiente para dar por bueno el
mecanismo, no para equiparar ambas cifras: Understat calcula sus puntos esperados con su
propio procedimiento y el fichero de remates esta limpiado.

Un recorte del 19,2 % equivale a borrar por completo los 7,4 goles esperados de exceso de
esa zona. Ahi la probabilidad de llegar a 43 puntos es del 61,7 %.

## 3. El techo del cambio

Un solo fichaje no cubre el problema entero, porque el refuerzo solo juega una parte de
los minutos de central. Este apartado acota cuanto puede aportar como maximo, usando los
minutos reales en lugar de suponer un efecto.

In [3]:
# Grupo de referencia: centrales de la liga con minutos suficientes
pool = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].copy()
# Se exigen las cuatro metricas defensivas completas, como en el trabajo
pool = pool.dropna(subset=["Tackle", "Interception", "Clearance", "Aerial"])
# Intercepciones por 90 minutos de cada central del grupo
pool["int90"] = pool["Interception"] / pool["minutes"] * 90
# Comprobacion del tamano del grupo de referencia
print(f"Grupo de referencia: {len(pool)} centrales")

# Centrales del Mallorca con minutos registrados
mal = m[(m["team"] == EQ) & (m["posicion"] == "central") & (m["minutes"] > 0)].copy()
# Intercepciones por 90 minutos de cada uno
mal["int90"] = mal["Interception"] / mal["minutes"] * 90
# Percentil de cada central dentro del grupo de referencia
mal["percentil"] = mal["int90"].apply(
    lambda v: (pool["int90"] < v).mean() * 100)
# Orden de mas a menos minutos jugados
mal = mal.sort_values("minutes", ascending=False)

# Tabla de los centrales del Mallorca
print(f"\n=== CENTRALES DEL {EQ.upper()} ===")
print(mal[["jugador", "minutes", "int90", "percentil"]].round(2).to_string(index=False))

# Minutos totales de central disponibles en una temporada: dos puestos, 38 partidos
MIN_TOTALES = 38 * 90 * 2
# Los tres centrales con mas minutos son los habituales
habituales = mal.head(3)
# El mas flojo de los habituales en anticipacion
flojo = habituales.loc[habituales["int90"].idxmin()]
# Peso de sus minutos sobre el total de minutos de central
cuota = flojo["minutes"] / MIN_TOTALES

# Resultado del calculo del techo
print(f"\nMas flojo de los habituales: {flojo['jugador']}")
print(f"  {flojo['minutes']:.0f} minutos, {flojo['int90']:.2f} intercepciones por 90, percentil {flojo['percentil']:.0f}")
print(f"  Cubre el {cuota*100:.1f} % de los minutos de central de la temporada")

# Recorte maximo atribuible al cambio, suponiendo el caso mas favorable posible
techo = 0.192 * cuota
print(f"\nTecho del escenario: recorte del {techo*100:.1f} %")
print("Supone el caso mas favorable: que en los minutos de ese central")
print("desapareciera por completo el exceso de la zona.")

# Version numerica de los recortes simulados para poder interpolar
esc_num = esc.copy()
esc_num["r"] = recortes
# Puntos medios interpolados en el techo del escenario
pts_techo = np.interp(techo, esc_num["r"], esc_num["media"])
# Probabilidad de permanencia interpolada en el techo del escenario
prob_techo = np.interp(techo, esc_num["r"], esc_num["prob_43"])

# Comparacion entre el escenario base y el techo del cambio
print(f"\n=== LECTURA FINAL ===")
print(f"Base:        {esc_num.iloc[0]['media']:.2f} puntos, {esc_num.iloc[0]['prob_43']:.1f} % de llegar a 43")
print(f"Techo:       {pts_techo:.2f} puntos, {prob_techo:.1f} % de llegar a 43")
print(f"Problema entero resuelto (19,2 %): {np.interp(0.192, esc_num['r'], esc_num['media']):.2f} puntos, "
      f"{np.interp(0.192, esc_num['r'], esc_num['prob_43']):.1f} %")

Grupo de referencia: 65 centrales

=== CENTRALES DEL MALLORCA ===
        jugador  minutes  int90  percentil
 Martin Valjent     3311   0.63       9.23
 Antonio Raíllo     2069   0.70      12.31
    David López      827   0.87      30.77
Marash Kumbulla      434   0.41       3.08

Mas flojo de los habituales: Martin Valjent
  3311 minutos, 0.63 intercepciones por 90, percentil 9
  Cubre el 48.4 % de los minutos de central de la temporada

Techo del escenario: recorte del 9.3 %
Supone el caso mas favorable: que en los minutos de ese central
desapareciera por completo el exceso de la zona.

=== LECTURA FINAL ===
Base:        40.65 puntos, 37.4 % de llegar a 43
Techo:       42.43 puntos, 49.1 % de llegar a 43
Problema entero resuelto (19,2 %): 44.40 puntos, 61.7 %


**Valjent es el central mas flojo en anticipacion y el que mas juega**: percentil 9 de los
65 centrales de la liga y el 48,4 % de todos los minutos de central de la temporada. Esa
cuota marca el techo del escenario en un recorte del 9,3 %.

| Escenario | Recorte | Puntos esperados | Probabilidad de 43 puntos |
|---|---|---|---|
| Temporada real | 0 % | 40,65 | 37,4 % |
| Techo del refuerzo | 9,3 % | 42,43 | 49,1 % |
| Carencia resuelta | 19,2 % | 44,40 | 61,7 % |

**Ni en el caso mas favorable el refuerzo garantiza la permanencia**: la probabilidad pasa
del 37,4 % al 49,1 %. Resolver la carencia al completo, que es mas de lo que aporta un
fichaje, la deja en el 61,7 %.

*El techo del 9,3 % es un supuesto, no una estimacion: asume que en los minutos de Valjent
desapareceria todo el exceso de la zona, repartido en proporcion a los minutos. Es el caso
mas favorable concebible y como tal hay que leerlo.*

## 4. La figura

Curva de probabilidad de permanencia segun el recorte, con los tres escenarios marcados.
En version clara para el documento y oscura con fondo solido.

In [4]:
# Librerias de visualizacion
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Carpeta de imagenes del proyecto
IMAGES = os.path.join(BASE, "images")
# Creacion de la carpeta si no existe
os.makedirs(IMAGES, exist_ok=True)

# Paleta de la version para documento
CLARO_FONDO, CLARO_TXT, CLARO_SUAVE = "#FFFFFF", "#1B2430", "#5A6675"
# Paleta de la version oscura
OSC_FONDO, OSC_TXT, OSC_SUAVE = "#0D1420", "#F2F5F7", "#C8CFD5"
# Color de acento, representativo del Mallorca
ROJO = "#E8485C"

# Valores de los tres puntos que se marcan en la figura
# El tercer valor indica la fila de rotulos: 0 la superior, 1 la inferior
hitos = [
    (0.0, "Temporada real", 1),
    (techo, "Techo del refuerzo", 0),
    (0.192, "Carencia resuelta", 1),
]
# Altura del nombre, de las cifras y del arranque de la guia en cada fila
FILA_NOMBRE, FILA_CIFRAS, FILA_GUIA = (-7, -22), (-14.5, -29.5), (-4.5, -19.5)


# Funcion que dibuja la figura en la paleta que se le indique
def dibujar(fondo, txt, suave, ruta, transparente):
    # Lienzo de la figura
    fig = plt.figure(figsize=(10.0, 6.2), facecolor=fondo)
    # Ejes con margenes amplios para que no se solape nada
    ax = fig.add_axes([0.085, 0.235, 0.890, 0.565])
    # Fondo de los ejes
    ax.set_facecolor(fondo)

    # Linea de referencia en el 50 % de probabilidad
    ax.axhline(50, color=suave, lw=0.9, ls=(0, (4, 4)), alpha=0.55, zorder=1)
    # Etiqueta de esa linea de referencia
    ax.text(61.5, 51.2, "moneda al aire", color=suave, fontsize=9.5,
            style="italic", va="bottom", ha="right", zorder=2)

    # Curva del barrido de escenarios
    ax.plot(esc_num["r"] * 100, esc_num["prob_43"], color=ROJO, lw=2.6,
            solid_capstyle="round", zorder=3)

    # Recorrido de los tres hitos marcados
    for i, (xr, etiqueta, fila) in enumerate(hitos):
        # Probabilidad interpolada en ese recorte
        yv = np.interp(xr, esc_num["r"], esc_num["prob_43"])
        # Puntos esperados interpolados en ese recorte
        pv = np.interp(xr, esc_num["r"], esc_num["media"])
        # Linea de guia desde la zona de rotulos hasta el punto
        ax.vlines(xr * 100, FILA_GUIA[fila], yv - 2.5,
                  color=suave, lw=0.9, alpha=0.45, zorder=2)
        # Punto marcado sobre la curva
        ax.plot(xr * 100, yv, "o", ms=10, color=ROJO,
                mec=fondo, mew=2.2, zorder=4)
        # El primer rotulo se alinea a la izquierda para no salirse del lienzo
        ha = "left" if i == 0 else "center"
        # Pequeno margen para que ese primer rotulo no toque el eje
        xt = xr * 100 + (0.8 if i == 0 else 0)
        # Nombre del escenario
        ax.text(xt, FILA_NOMBRE[fila], etiqueta, ha=ha, va="bottom", color=txt,
                fontsize=11.5, fontweight="bold", zorder=5)
        # Cifras del escenario debajo del nombre
        ax.text(xt, FILA_CIFRAS[fila], f"{yv:.0f} %  ·  {pv:.1f} pts",
                ha=ha, va="bottom", color=suave, fontsize=10, zorder=5)

    # Limites del eje horizontal
    ax.set_xlim(-1.5, 62)
    # Limites del eje vertical, con espacio inferior para los rotulos
    ax.set_ylim(-36, 100)
    # Marcas del eje horizontal cada diez puntos
    ax.set_xticks(range(0, 70, 10))
    # Marcas del eje vertical cada veinte puntos
    ax.set_yticks(range(0, 101, 20))
    # Formato de porcentaje en el eje horizontal
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f} %"))
    # Formato de porcentaje en el eje vertical
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f} %"))
    # Rotulo del eje horizontal
    ax.set_xlabel("Recorte del xG concedido en el centro del área en juego en curso",
                  color=suave, fontsize=11.5, labelpad=10)
    # Rotulo del eje vertical
    ax.set_ylabel("Probabilidad de alcanzar 43 puntos", color=suave,
                  fontsize=11.5, labelpad=10)
    # Color y tamano de los numeros de los ejes
    ax.tick_params(colors=suave, labelsize=10.5, length=0)
    # Rejilla horizontal suave
    ax.grid(axis="y", color=suave, alpha=0.16, lw=0.8)
    # La rejilla se dibuja por detras de los datos
    ax.set_axisbelow(True)
    # Se eliminan los bordes del recuadro
    for lado in ax.spines.values():
        lado.set_visible(False)

    # Titulo de la figura
    fig.text(0.085, 0.962, "El refuerzo mejora la permanencia, no la garantiza",
             color=txt, fontsize=18, fontweight="bold", va="top")
    # Subtitulo con la lectura principal
    fig.text(0.085, 0.884,
             "Sustituir al central más flojo sube la probabilidad del 37 % al 49 %. "
             "Resolver la carencia\nal completo, más de lo que aporta un fichaje, la deja en el 62 %.",
             color=suave, fontsize=12, va="top", linespacing=1.5)

    # Pie con la fuente de los datos
    fig.text(0.085, 0.118,
             "Elaboración propia a partir de los remates de LaLiga 2025/26 (Understat) "
             "y los registros defensivos individuales (WhoScored).",
             color=suave, fontsize=9.2, va="top")
    # Pie con el metodo y el supuesto declarado
    fig.text(0.085, 0.080,
             "20.000 temporadas simuladas remate a remate por escenario. Umbral de permanencia: "
             "43 puntos, con el resto de la liga fijo.\nEs un análisis de sensibilidad, no una predicción: "
             "el recorte es un supuesto y no una estimación del efecto del fichaje.",
             color=suave, fontsize=9.2, va="top", linespacing=1.45)

    # Guardado de la figura
    fig.savefig(ruta, dpi=200, facecolor="none" if transparente else fondo,
                transparent=transparente, bbox_inches=None, pad_inches=0)
    # Cierre de la figura para liberar memoria
    plt.close(fig)
    # Confirmacion de la ruta de guardado
    print("Guardada:", ruta)


# Version clara, la que va en el README
dibujar(CLARO_FONDO, CLARO_TXT, CLARO_SUAVE,
        os.path.join(IMAGES, "simulacion_contrafactual.png"), False)
# Version oscura con fondo solido, valida en cualquier sitio
dibujar(OSC_FONDO, OSC_TXT, OSC_SUAVE,
        os.path.join(IMAGES, "simulacion_contrafactual_oscura.png"), False)

Guardada: ..\images\simulacion_contrafactual.png


Guardada: ..\images\simulacion_contrafactual_oscura.png


**Figura 8.** Probabilidad de que el RCD Mallorca alcanzara los 43 puntos según el
recorte aplicado al xG concedido en el centro de su área en juego en curso. Sustituir al
central más flojo sube la probabilidad del 37 % al 49 %; resolver la carencia al completo
la deja en el 62 %.

*Fuente: elaboración propia a partir de los remates con xG de Understat y los registros
defensivos individuales de WhoScored. 20.000 temporadas simuladas por escenario.*

---
## Conclusiones

1. **El efecto del refuerzo no es estimable con estos datos.** La regresion a nivel de
   equipo no encuentra relacion (R2 0,040, p 0,397), igual que el modelo supervisado del
   notebook 04. Lo que hay aqui es un analisis de sensibilidad, no una prediccion.
2. **La carencia tuvo un coste real**: unos 12 puntos porcentuales de probabilidad de
   permanencia en el escenario mas favorable, y unos 25 si se resolviera por completo.
3. **Pero no fue causa suficiente del descenso.** El equipo sumo ademas 2,21 puntos mas de
   los que su juego merecia y bajo igualmente.

### Limitaciones

- El recorte es un supuesto declarado, no una estimacion del efecto del fichaje.
- El techo del 9,3 % reparte el exceso en proporcion a los minutos, lo cual no es
  demostrable.
- Los resultados del resto de la liga se mantienen fijos.
- No se recogen efectos indirectos de un cambio de jugador sobre el comportamiento
  colectivo del equipo.
- Las cifras simuladas no son comparables con los 42 puntos reales, solo entre si.